# stage4 第二輪重訓 bi-encoder — Colab GPU

用第二輪「破模板」訓練集(含跨域類比,`generalization_queries.json`)重訓 bi-encoder,
在 holdout(生成時隔離、絕不進訓練)上驗證**真泛化** —— 不再被 selection-bias 數字自欺。

**先做**:Runtime → Change runtime type → **GPU**(T4 即可)。

閉環:破模板訓練集 → 重訓 → export onnx → 重編房源向量 → holdout/unified 評估。

> **同源鐵則**:重訓出的是新 bi-encoder → 房源向量必須用它重編(舊向量作廢),
> 且前端 onnx 也要換成這顆。三者(onnx + 房源向量 + 模型)必須同批。

> **守天花板**:stage4 unified 0.739 是已知天花板(向量塌縮教訓)。重訓後對
> holdout + unified 評估,**沒進步就不落地**(誠實:破模板不保證一定贏)。

## 0. 環境 — clone main(含第二輪資料)+ 裝套件

In [ ]:
!nvidia-smi -L
%cd /content
![ -d nchu-edge-rental-ai ] || git clone https://github.com/eric20041027/nchu-edge-rental-ai.git
%cd /content/nchu-edge-rental-ai
!git checkout main && git pull
!pip -q install 'transformers<5' tokenizers onnx onnxruntime
import torch, json; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
d = json.load(open('data/processed/generalization_queries.json'))
cross = [r for r in d if r.get('feat') and any(x in r['query'] for x in ['校園宿舍','便利商店','銅板','泡麵','飯店','圖書館','百貨','深山'])]
print('訓練 query:', len(d), '| 跨域類比:', len(cross), '(第二輪破模板)')

## 1. 重訓 bi-encoder(餵第二輪破模板訓練集)

關鍵:用 `TRAIN_DATA_PATH` env 指向 generalization_queries.json(train_bi_encoder
預設讀 recommendation_train.json,需覆蓋)。schema 已驗相容(query/property/label/is_hard)。

In [ ]:
import os
os.environ['TRAIN_DATA_PATH'] = 'data/processed/generalization_queries.json'
!TRAIN_DATA_PATH=data/processed/generalization_queries.json \
    python -m pipeline.model_training.train_bi_encoder \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-length 64 --bf16 --tf32 \
    --output-dir saved_models/rbt6_bi_encoder_r2
# 觀察 log:dev recall_at_1_vs_pool;loss 收斂。
# (此為 6 層 rbt6 重訓。若要 rbt3 student,再跑 distill_bi_encoder,見蒸餾 notebook。)

## 2. (可選)蒸餾成 rbt3 student

現役前端是 rbt3(38MB)。要維持體積,把 step1 的 rbt6 蒸成 rbt3。
若只想先驗「破模板有沒有用」可跳過,直接用 rbt6 評估(step 3 改 --saved-dir 指 r2)。

In [ ]:
!TRAIN_DATA_PATH=data/processed/generalization_queries.json \
    python -m pipeline.model_training.distill_bi_encoder \
    --teacher-dir saved_models/rbt6_bi_encoder_r2 \
    --output-dir saved_models/rbt3_bi_encoder_r2 \
    --epochs 3 --batch-size 32 --lr 3e-5 --alpha 0.5 --max-length 64 --bf16 --tf32

## 3. Export onnx + 重編房源向量(同源)

STUDENT 改成你要落地的那顆(rbt3_bi_encoder_r2 或 rbt6_bi_encoder_r2)。

In [ ]:
STUDENT = 'saved_models/rbt3_bi_encoder_r2'  # ← 若跳過 step2 改 rbt6_bi_encoder_r2
!python -m pipeline.model_training.export_bi_encoder --saved-dir {STUDENT}
!python -m pipeline.data_prep.build_property_embeddings --saved-dir {STUDENT}
!python -m pipeline.data_prep.build_property_embeddings --check

## 4. 驗真泛化 — holdout + unified(守天花板)

holdout = 生成時隔離、風格刻意不同、絕不進訓練 → 真泛化試金石。
unified = 974 房源客觀 GT(批次後 precision/recall 分桶)。

In [ ]:
print('=== holdout(真泛化) ===')
!python pipeline/evaluation/eval_generalization.py --eval-set tests/fixtures/generalization_holdout.json
print('=== unified(974 客觀) ===')
!python pipeline/evaluation/eval_generalization.py --unified
print('=== A/B vs rule-based(recall gate) ===')
!python pipeline/evaluation/eval_vector_vs_rulebased.py

## 5. Gate + 打包下載

| 指標 | 門檻 |
|---|---|
| holdout Recall/Precision | **> 重訓前**(真泛化進步) |
| unified 總分 | ≥ 重訓前(0.196 baseline,別退步) |
| eval_vector_vs_rulebased | 仍 GO |
| onnx 大小 | rbt3 ~38MB / rbt6 ~57MB |

**沒進步就不落地** — 誠實面對「破模板不保證贏」(stage4 0.739 天花板)。
全過才下載三檔(onnx + 重編向量),版本同批。

In [ ]:
import os, json
sz = os.path.getsize('frontend/models/bi_encoder_dir/bi_encoder_quant.onnx')/1e6
emb = json.load(open('frontend/assets/property_embeddings.json'))
pd = len(json.load(open('frontend/assets/property_data.json')))
print(f'onnx={sz:.1f}MB | embeddings={emb["count"]} vs property_data={pd}')
assert emb['count'] == pd, '向量數 != 房源數!同源錯配'
!cd /content/nchu-edge-rental-ai && tar czf /content/stage4_r2_artifacts.tgz \
    frontend/models/bi_encoder_dir/bi_encoder_quant.onnx \
    frontend/assets/property_embeddings.json
from google.colab import files; files.download('/content/stage4_r2_artifacts.tgz')

## 落地(本機,gate 全過才做)

```bash
cd <repo>
git checkout -b claude/stage4-r2-retrain-land
tar xzf ~/Downloads/stage4_r2_artifacts.tgz   # 覆蓋 onnx + 房源向量
git add frontend/models/bi_encoder_dir/bi_encoder_quant.onnx frontend/assets/property_embeddings.json
git commit -m 'feat(stage4-r2): 破模板重訓 bi-encoder 落地(holdout 真泛化↑)'
```

把 step4 的 holdout / unified / GO 數字貼回對話,一起對 gate 驗收再開 PR。
⚠️ onnx 與 property_embeddings 必須同批(同一次重訓產出),否則召回壞。